# Entraînement wake word « bilou » — pipeline propre

Pipeline minimale et robuste : génération piper (nouvelle API) → embeddings
ONNX → petit classifieur PyTorch → export .onnx.

**Aucune** dépendance fragile (torchmetrics / speechbrain / torch-audiomentations
/ tensorflow). Tourne sur le Colab actuel (py3.12) sans épinglage de vieilles
versions.

Runtime conseillé : **GPU** (Exécution → Modifier le type d''exécution → T4).

In [ ]:
# 1. Récupère le repo de la pipeline
!git clone https://github.com/Epwo/bilou-wakeword.git
%cd bilou-wakeword

In [ ]:
# 2. Installe les dépendances (saines, non épinglées)
# NB: piper-sample-generator n'est PAS pip-installé (wheel cassé) — il est
# cloné automatiquement par la pipeline. On installe juste le reste.
!pip install -q openwakeword onnxruntime soundfile scipy
# torch/numpy sont déjà dans Colab

In [ ]:
# 3. Télécharge la voix française + les features négatives pré-calculées
!python download_data.py

In [ ]:
# 4. (optionnel) Écoute un échantillon de "bilou" avant d'entraîner
from pipeline.generate import generate_positive_samples
generate_positive_samples("bilou", "work/test", "voices/fr_FR-upmc-medium.onnx", n_samples=1)
from IPython.display import Audio
import glob
Audio(sorted(glob.glob("work/test/*.wav"))[0], autoplay=True)

In [ ]:
# 5. Lance toute la pipeline (génération + features + entraînement + export)
#    ~10-20 min sur GPU pour 2000 samples.
!python run_all.py --word bilou --n-samples 2000 --epochs 20

In [ ]:
# 6. Télécharge le modèle entraîné
from google.colab import files
files.download("models/bilou.onnx")

## Ensuite, côté Reachy

```bash
mv ~/Downloads/bilou.onnx voice_agent/wake/models/bilou.onnx
source .venv_wake/bin/activate
python wake/wake_word.py --model wake/models/bilou.onnx --meter
```